# Project 1: Initial Summary

This project involves the application of predictive modeling and machine learning techniques to a real-world dataset. The main objectives and tasks are as follows:

- **Understand the Problem Statement:**  
  Carefully review the dataset and the prediction task described in the project document.

- **Data Exploration and Preprocessing:**  
  - Load and inspect the dataset.
  - Handle missing values, outliers, and perform necessary data cleaning.
  - Conduct exploratory data analysis (EDA) to understand feature distributions and relationships.

- **Feature Engineering:**  
  - Create or transform features to improve model performance.
  - Encode categorical variables and scale numerical features as needed.

- **Model Selection and Training:**  
  - Select appropriate machine learning algorithms for the prediction task (e.g., regression, classification).
  - Train and validate models using suitable evaluation metrics.

- **Model Evaluation and Tuning:**  
  - Assess model performance using cross-validation and test data.
  - Tune hyperparameters to optimize results.

- **Interpretation and Reporting:**  
  - Interpret the results and provide insights based on model outputs.
  - Summarize findings, challenges, and recommendations.

- **Documentation:**  
  - Document all steps, code, and results clearly in the notebook.
  - Prepare a final report as required by the project guidelines.

## Step 1: Load all Libraries


In [1]:
# ============================================================
# Import all required libraries for link prediction pipeline
# ============================================================

# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Text parsing and randomization utilities
import re
import random

# Efficient dictionary with default values for graph adjacency
from collections import defaultdict

# Mathematical functions for Adamic-Adar index computation
from math import log

# Machine learning: train/test splitting, feature scaling,
# logistic regression classifier, pipeline orchestration, evaluation metric
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


### Step 2: Configuration & Reproducibility

**Summary:** Set random seeds (`random`, `numpy`) to **42** for reproducible results across runs. Define `POSITIVE_EDGE_SAMPLE_LIMIT` — the maximum number of real edges to sample for training. This controls the trade-off between training time/memory and model accuracy. Increasing this value on machines with more RAM improves performance.

In [2]:
# ============================================================
# Step 0: Set random seeds for reproducibility and define
#         the maximum number of positive edge samples to use
# ============================================================

# Fix random seeds so results are reproducible across runs
random.seed(42)
np.random.seed(42)

# Maximum number of positive (real) edges to sample for training.
# Increase this value on machines with more RAM/CPU for better accuracy.
POSITIVE_EDGE_SAMPLE_LIMIT = 10000  # try 40_000 on a beefier machine


### Step 3: Parse Training Graph & Build Adjacency Structures

**Summary:** Read `train.csv` and construct the directed graph as adjacency lists (`outgoing_neighbors`, `incoming_neighbors`). Collect all unique node IDs and directed edges. Then build **undirected neighbor sets** (union of in/out edges) and precompute **degree dictionaries** (`out_degree_map`, `in_degree_map`, `undirected_degree_map`) for fast feature lookups in later cells. Prints graph statistics (|V|, |E|, average degrees).

In [3]:
# ============================================================
# Step 1: Parse the training CSV file and build the directed
#         graph adjacency structure (outgoing + incoming neighbors)
# ============================================================

def parse_training_graph_from_csv(csv_file_path):
    """
    Read the training CSV and construct directed graph adjacency lists.

    Each row in the CSV has the format: source_node, dest1, dest2, ...
    This function builds:
      - outgoing_neighbors: {node -> set of nodes it points to}
      - incoming_neighbors: {node -> set of nodes that point to it}
      - all_graph_nodes: set of all unique node IDs
      - directed_edge_list: list of (source, destination) tuples

    Returns:
        tuple: (outgoing_neighbors, incoming_neighbors, all_graph_nodes, directed_edge_list)
    """
    outgoing_neighbors = defaultdict(set)
    incoming_neighbors = defaultdict(set)
    all_graph_nodes = set()
    directed_edge_list = []

    with open(csv_file_path, 'r', encoding='utf-8') as file_handle:
        for raw_line in file_handle:
            raw_line = raw_line.strip()
            if not raw_line:
                continue
            # Skip any truncation footer that may appear in previews
            if 'This Output has been Truncated' in raw_line:
                break

            # Split each line by comma or whitespace to extract node IDs
            token_parts = re.split(r'[\\s,]+', raw_line)
            parsed_node_ids = []
            for token in token_parts:
                if not token:
                    continue
                try:
                    parsed_node_ids.append(int(token))
                except ValueError:
                    pass

            # Need at least a source + one destination
            if len(parsed_node_ids) <= 1:
                continue

            source_node = parsed_node_ids[0]
            for destination_node in parsed_node_ids[1:]:
                # Skip self-loops
                if destination_node == source_node:
                    continue
                # Only add the edge if it doesn't already exist
                if destination_node not in outgoing_neighbors[source_node]:
                    outgoing_neighbors[source_node].add(destination_node)
                    incoming_neighbors[destination_node].add(source_node)
                    directed_edge_list.append((source_node, destination_node))
                    all_graph_nodes.add(source_node)
                    all_graph_nodes.add(destination_node)

    return outgoing_neighbors, incoming_neighbors, all_graph_nodes, directed_edge_list


# Build the full directed graph from train.csv
outgoing_neighbors, incoming_neighbors, all_graph_nodes, directed_edge_list = parse_training_graph_from_csv('train.csv')
print(f"Graph: |V|={len(all_graph_nodes):,}, |E|={len(directed_edge_list):,}")

# ── Build undirected neighbor sets by merging outgoing and incoming edges ──
# This is used for undirected similarity metrics (Jaccard, Adamic-Adar, etc.)
undirected_neighbors = defaultdict(set)
for node_id in all_graph_nodes:
    if node_id in outgoing_neighbors:
        undirected_neighbors[node_id] |= outgoing_neighbors[node_id]
    if node_id in incoming_neighbors:
        undirected_neighbors[node_id] |= incoming_neighbors[node_id]

# ── Precompute degree dictionaries for fast feature lookups ──
# out_degree_map: number of outgoing edges per node
# in_degree_map: number of incoming edges per node
# undirected_degree_map: total unique neighbors (ignoring direction)
out_degree_map = {node_id: len(outgoing_neighbors[node_id]) for node_id in all_graph_nodes}
in_degree_map = {node_id: len(incoming_neighbors[node_id]) for node_id in all_graph_nodes}
undirected_degree_map = {node_id: len(undirected_neighbors[node_id]) for node_id in all_graph_nodes}

print(f"Avg out-degree: {np.mean(list(out_degree_map.values())):.2f}")
print(f"Avg in-degree: {np.mean(list(in_degree_map.values())):.2f}")
print(f"Avg undirected degree: {np.mean(list(undirected_degree_map.values())):.2f}")


Graph: |V|=4,867,136, |E|=23,945,585
Avg out-degree: 4.92
Avg in-degree: 4.92
Avg undirected degree: 9.62


### Step 4: Define Basic Link-Prediction Feature Function

**Summary:** Define `compute_basic_link_features(source, target)` which computes **13 hand-crafted structural features** for a given node pair:

| # | Feature | Description |
|---|---------|-------------|
| 1–4 | Degree features | Source out-degree, target in-degree, undirected degrees |
| 5–8 | Log-degree features | `log1p` transforms of the above four |
| 9 | Common neighbors | Count of shared undirected neighbors |
| 10 | Jaccard similarity | Common / union neighbor ratio |
| 11 | Adamic-Adar index | Weighted sum over common neighbors (1/log(degree)) |
| 12 | Preferential attachment | out_degree(src) × in_degree(tgt) |
| 13 | Reverse edge flag | Whether the reverse edge (tgt→src) exists |

Also defines `BASIC_FEATURE_COLUMN_NAMES` for model interpretation.

In [4]:
# ============================================================
# Step 2: Compute lightweight link-prediction features for a
#         given (source, target) node pair
# ============================================================

def compute_basic_link_features(source_node, target_node):
    """
    Compute 13 hand-crafted structural features for link prediction
    between source_node and target_node.

    Features include:
      - Raw and log-transformed degrees (out, in, undirected)
      - Common neighbors count (undirected)
      - Jaccard similarity (undirected)
      - Adamic-Adar index (undirected)
      - Preferential attachment score (directed: out * in)
      - Reverse edge indicator (does target→source edge exist?)

    Args:
        source_node (int): The source node ID
        target_node (int): The target node ID

    Returns:
        list: A list of 13 numeric feature values
    """
    # Retrieve undirected neighbor sets for both nodes
    source_undirected_neighbors = undirected_neighbors.get(source_node, set())
    target_undirected_neighbors = undirected_neighbors.get(target_node, set())

    # Retrieve degree values from precomputed dictionaries
    source_out_degree = out_degree_map.get(source_node, 0)
    target_in_degree = in_degree_map.get(target_node, 0)
    source_undirected_degree = undirected_degree_map.get(source_node, 0)
    target_undirected_degree = undirected_degree_map.get(target_node, 0)

    # Common neighbors: nodes connected to both source and target (undirected)
    common_neighbor_set = source_undirected_neighbors & target_undirected_neighbors
    common_neighbor_count = len(common_neighbor_set)

    # Neighborhood union size for Jaccard denominator
    neighborhood_union_size = len(source_undirected_neighbors | target_undirected_neighbors)
    jaccard_similarity = common_neighbor_count / neighborhood_union_size if neighborhood_union_size > 0 else 0.0

    # Adamic-Adar index: sum of 1/log(degree) for each common neighbor
    # Higher weight for common neighbors with fewer connections (more informative)
    adamic_adar_score = 0.0
    for shared_neighbor in common_neighbor_set:
        neighbor_degree = undirected_degree_map.get(shared_neighbor, 0)
        if neighbor_degree > 1:
            adamic_adar_score += 1.0 / log(neighbor_degree)

    # Preferential attachment: product of source out-degree and target in-degree
    # Based on the intuition that high-degree nodes attract more connections
    preferential_attachment = source_out_degree * target_in_degree

    # Check if the reverse edge (target → source) already exists
    reverse_edge_exists = 1 if source_node in outgoing_neighbors.get(target_node, set()) else 0

    return [
        source_out_degree, target_in_degree,
        source_undirected_degree, target_undirected_degree,
        np.log1p(source_out_degree), np.log1p(target_in_degree),
        np.log1p(source_undirected_degree), np.log1p(target_undirected_degree),
        common_neighbor_count, jaccard_similarity,
        adamic_adar_score, preferential_attachment, reverse_edge_exists
    ]


# Descriptive names for each of the 13 features (used for model interpretation)
BASIC_FEATURE_COLUMN_NAMES = [
    'source_out_degree', 'target_in_degree',
    'source_undirected_degree', 'target_undirected_degree',
    'log_source_out_degree', 'log_target_in_degree',
    'log_source_undirected_degree', 'log_target_undirected_degree',
    'common_neighbor_count', 'jaccard_similarity',
    'adamic_adar_index', 'preferential_attachment', 'reverse_edge_flag'
]


### Step 5: Negative Sampling & Train/Validation Split

**Summary:** Create labeled training data for link prediction:

1. **Positive samples** — Shuffle real edges from the graph and cap at `POSITIVE_EDGE_SAMPLE_LIMIT`.
2. **Negative samples** — Randomly generate node pairs that have **no edge** between them, ensuring a **balanced** 1:1 class ratio.
3. **Train/Val split** — Split both positive and negative samples 80/20 using `train_test_split(random_state=42)`.
4. **Feature extraction** — Compute the 13 basic structural features for all labeled pairs, producing `features_train`, `labels_train`, `features_validation`, `labels_validation` numpy arrays.

In [5]:
# ============================================================
# Step 3: Create labeled training data using negative sampling
#
# Positive samples = real edges from the graph
# Negative samples = randomly generated node pairs that have
#                    no edge between them (non-edges)
# ============================================================

# ── Sample positive edges (real links) ──
positive_edge_samples = directed_edge_list.copy()
random.shuffle(positive_edge_samples)
# Cap the number of positive samples to control training size
if len(positive_edge_samples) > POSITIVE_EDGE_SAMPLE_LIMIT:
    positive_edge_samples = positive_edge_samples[:POSITIVE_EDGE_SAMPLE_LIMIT]

# ── Generate negative samples (non-existent links) ──
# Randomly pair nodes that are NOT connected, ensuring equal class balance
all_nodes_as_list = list(all_graph_nodes)
negative_edge_samples = set()
while len(negative_edge_samples) < len(positive_edge_samples):
    random_source = random.choice(all_nodes_as_list)
    random_target = random.choice(all_nodes_as_list)
    # Skip self-loops
    if random_source == random_target:
        continue
    # Only add if no real edge exists from source to target
    if random_target not in outgoing_neighbors.get(random_source, set()):
        negative_edge_samples.add((random_source, random_target))
negative_edge_samples = list(negative_edge_samples)

# ── Split into training and validation sets (80/20) ──
positive_train, positive_validation = train_test_split(
    positive_edge_samples, test_size=0.2, random_state=42
)
negative_train, negative_validation = train_test_split(
    negative_edge_samples, test_size=0.2, random_state=42
)

# Combine positives (label=1) and negatives (label=0) into labeled pairs
labeled_train_pairs = (
    [(src, tgt, 1) for (src, tgt) in positive_train] +
    [(src, tgt, 0) for (src, tgt) in negative_train]
)
labeled_validation_pairs = (
    [(src, tgt, 1) for (src, tgt) in positive_validation] +
    [(src, tgt, 0) for (src, tgt) in negative_validation]
)

# ── Build feature matrices and label vectors ──
# Each row = 13 structural features computed for one (source, target) pair
features_train = np.array(
    [compute_basic_link_features(src, tgt) for (src, tgt, _) in labeled_train_pairs],
    dtype=float
)
labels_train = np.array(
    [label for (_, _, label) in labeled_train_pairs], dtype=int
)

features_validation = np.array(
    [compute_basic_link_features(src, tgt) for (src, tgt, _) in labeled_validation_pairs],
    dtype=float
)
labels_validation = np.array(
    [label for (_, _, label) in labeled_validation_pairs], dtype=int
)


## Model 1: Logistic Regression

### Step 6: Train Logistic Regression Pipeline

**Summary:** Build a `sklearn` Pipeline with **StandardScaler → LogisticRegression** (balanced class weights, LBFGS solver, max_iter=1000). Train on the feature matrix, evaluate on validation set using **AUC-ROC**, then refit on **all labeled data** (train + val combined) for final predictions.

In [6]:
# ============================================================
# Step 4: Train a Logistic Regression model inside a Pipeline
#         (StandardScaler + LogisticRegression)
# ============================================================

# Build a pipeline that first normalizes features (zero mean, unit variance)
# then fits a logistic regression with balanced class weights
logistic_regression_pipeline = Pipeline([
    ('feature_scaler', StandardScaler()),
    ('logistic_classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # up-weight the minority class
        solver='lbfgs'            # good default for small-medium datasets
    ))
])

# Train on the training split
logistic_regression_pipeline.fit(features_train, labels_train)

# Evaluate on the validation split using AUC-ROC
# predict_proba[:,1] gives the probability of the positive class (link exists)
validation_auc_score = roc_auc_score(
    labels_validation,
    logistic_regression_pipeline.predict_proba(features_validation)[:, 1]
)
print(f"Validation AUC: {validation_auc_score:.4f}")

# ── Refit the model on ALL labeled data (train + val) for final predictions ──
features_combined = np.vstack([features_train, features_validation])
labels_combined = np.concatenate([labels_train, labels_validation])
logistic_regression_pipeline.fit(features_combined, labels_combined)


Validation AUC: 1.0000


,steps,"[('feature_scaler', ...), ('logistic_classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


### Step 7: Hyperparameter Tuning via GridSearchCV

**Summary:** Perform **5-fold cross-validated grid search** over Logistic Regression hyperparameters:
- `C` ∈ {0.01, 0.1, 1, 10, 100} — inverse regularization strength
- `solver` ∈ {lbfgs, liblinear}
- `max_iter` ∈ {500, 1000, 2000}

Uses `roc_auc` scoring with `n_jobs=-1` for parallel execution. After finding the best parameters, evaluates on validation set, then refits the best estimator on **all labeled data** for final predictions.

In [7]:
# ============================================================
# Step 4b: Hyperparameter tuning via GridSearchCV
#          (no feature selection — pure LogisticRegression tuning)
# ============================================================
from sklearn.model_selection import GridSearchCV

# Define the hyperparameter search space for LogisticRegression
# C = inverse regularization strength; solver = optimization algorithm
logistic_regression_param_grid = {
    'logistic_classifier__C': [0.01, 0.1, 1, 10, 100],
    'logistic_classifier__solver': ['lbfgs', 'liblinear'],
    'logistic_classifier__max_iter': [500, 1000, 2000]
}

# Pipeline: scale features first, then classify
tuning_pipeline = Pipeline([
    ('feature_scaler', StandardScaler()),
    ('logistic_classifier', LogisticRegression(class_weight='balanced'))
])

# Run 5-fold cross-validated grid search, scoring by AUC-ROC
grid_search_cv = GridSearchCV(
    tuning_pipeline,
    logistic_regression_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1  # use all available CPU cores
)
grid_search_cv.fit(features_train, labels_train)

print(f"Best parameters: {grid_search_cv.best_params_}")
print(f"Best CV AUC: {grid_search_cv.best_score_:.4f}")

# ── Evaluate the best model on the held-out validation set ──
tuned_validation_auc = roc_auc_score(
    labels_validation,
    grid_search_cv.predict_proba(features_validation)[:, 1]
)
print(f"Validation AUC after tuning: {tuned_validation_auc:.4f}")

# ── Refit the best estimator on ALL labeled data for final predictions ──
features_combined = np.vstack([features_train, features_validation])
labels_combined = np.concatenate([labels_train, labels_validation])
grid_search_cv.best_estimator_.fit(features_combined, labels_combined)


Best parameters: {'logistic_classifier__C': 0.01, 'logistic_classifier__max_iter': 500, 'logistic_classifier__solver': 'liblinear'}
Best CV AUC: 0.9999
Validation AUC after tuning: 1.0000


,steps,"[('feature_scaler', ...), ('logistic_classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.01


### Step 8: Score Test Set & Generate Submission

**Summary:** Load `test.csv` with (Id, From, To) triples. Compute the same 13 structural features for each test edge. Use the trained Logistic Regression pipeline to predict **link probabilities** (`predict_proba[:,1]`). Save the results to `submission.csv` with columns `Id` and `Predictions`.

In [8]:
# ============================================================
# Step 5: Score the test set and generate submission file
# ============================================================

# Load the test CSV containing (Id, From, To) triples, sorted by Id
test_dataframe = pd.read_csv('test.csv').sort_values('Id')

# Compute the same 13 structural features for every test edge
test_feature_matrix = np.array(
    [compute_basic_link_features(int(row.From), int(row.To))
     for row in test_dataframe.itertuples(index=False)],
    dtype=float
)

# Predict link probabilities using the trained pipeline
test_link_probabilities = logistic_regression_pipeline.predict_proba(test_feature_matrix)[:, 1]

# Build and save the submission DataFrame
submission_dataframe = pd.DataFrame({
    'Id': test_dataframe['Id'].values,
    'Predictions': test_link_probabilities
})
submission_dataframe.to_csv('submission.csv', index=False)
print("Saved: submission.csv")
print(submission_dataframe.head(10))


Saved: submission.csv
   Id  Predictions
0   1     0.999964
1   2     0.965663
2   3     0.999988
3   4     0.997606
4   5     0.999932
5   6     0.997168
6   7     0.993821
7   8     0.999799
8   9     0.997199
9  10     0.991587


## Model 1 : Bayesian Graph Neural Network (BGNN) Summary

A Bayesian Graph Neural Network (BGNN) combines the power of Graph Neural Networks (GNNs) with Bayesian inference. GNNs are designed to learn from graph-structured data, capturing relationships between nodes and edges. By introducing Bayesian layers, BGNNs model uncertainty in predictions, which is valuable for robust decision-making and understanding model confidence.

**Key Points:**
- BGNNs use probabilistic layers to estimate uncertainty.
- They are suitable for tasks like node classification, link prediction, and graph classification.
- Hyperparameter tuning (e.g., with Optuna) helps optimize model performance.
- BGNNs are especially useful when data is noisy or limited, as they provide uncertainty estimates alongside predictions.

In this notebook, we:
- Built a BGNN using PyTorch Geometric and torchbnn.
- Tuned hyperparameters for best performance.
- Evaluated the model and reported accuracy.

Replace the example graph data with your own for practical applications.

In [9]:
# ============================================================
# Bayesian Graph Neural Network (BGNN) for Link Prediction
#
# This cell implements a GCN with Bayesian linear layers for
# uncertainty-aware link prediction. It uses:
#   - PyTorch Geometric GCNConv for message passing
#   - torchbnn BayesLinear for probabilistic edge classification
#   - Optuna for hyperparameter optimization
# ============================================================

import torch
import torch_geometric.nn as pyg_nn
import torchbnn as bnn
import optuna
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split

# ── Select GPU if available, otherwise fall back to CPU ──
compute_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', compute_device)

# ============================================================
# Re-parse train.csv to build the graph adjacency for BGNN
# ============================================================
bgnn_outgoing_neighbors = defaultdict(set)
bgnn_incoming_neighbors = defaultdict(set)
bgnn_graph_nodes = set()
bgnn_edge_list = []

with open('train.csv', 'r', encoding='utf-8') as csv_file:
    for raw_line in csv_file:
        raw_line = raw_line.strip()
        if not raw_line or 'This Output has been Truncated' in raw_line:
            continue
        token_parts = raw_line.split(',')
        parsed_ids = [int(token) for token in token_parts if token.isdigit()]
        if len(parsed_ids) <= 1:
            continue
        source_node_id = parsed_ids[0]
        for dest_node_id in parsed_ids[1:]:
            if dest_node_id == source_node_id:
                continue
            bgnn_outgoing_neighbors[source_node_id].add(dest_node_id)
            bgnn_incoming_neighbors[dest_node_id].add(source_node_id)
            bgnn_edge_list.append((source_node_id, dest_node_id))
            bgnn_graph_nodes.add(source_node_id)
            bgnn_graph_nodes.add(dest_node_id)

# ── Create a sorted node list and a mapping from node ID → integer index ──
sorted_node_list = sorted(list(bgnn_graph_nodes))
node_id_to_index = {node_id: idx for idx, node_id in enumerate(sorted_node_list)}

# ── Build node feature matrix: [out_degree, in_degree, log(1+out), log(1+in)] ──
node_feature_matrix = np.zeros((len(sorted_node_list), 4))
for node_id in sorted_node_list:
    node_idx = node_id_to_index[node_id]
    node_out_degree = len(bgnn_outgoing_neighbors[node_id])
    node_in_degree = len(bgnn_incoming_neighbors[node_id])
    node_feature_matrix[node_idx, 0] = node_out_degree
    node_feature_matrix[node_idx, 1] = node_in_degree
    node_feature_matrix[node_idx, 2] = np.log1p(node_out_degree)
    node_feature_matrix[node_idx, 3] = np.log1p(node_in_degree)

# ── Build the edge index tensor for PyTorch Geometric (COO format) ──
# Shape: [2, num_edges] — first row = source indices, second row = target indices
graph_edge_index = torch.tensor(
    [[node_id_to_index[src] for src, dst in bgnn_edge_list],
     [node_id_to_index[dst] for src, dst in bgnn_edge_list]],
    dtype=torch.long
)

# ============================================================
# Negative sampling: generate non-edges for balanced training
# ============================================================
bgnn_positive_edges = bgnn_edge_list.copy()
max_edge_cap = min(10000, len(bgnn_positive_edges))  # limit for computational speed
bgnn_positive_edges = bgnn_positive_edges[:max_edge_cap]

bgnn_negative_edges = set()
existing_edge_set = set(bgnn_positive_edges)
all_nodes_array = np.array(sorted_node_list)

# Keep sampling random node pairs until we have enough non-edges
while len(bgnn_negative_edges) < len(bgnn_positive_edges):
    random_pair = tuple(np.random.choice(all_nodes_array, 2, replace=False))
    if random_pair not in existing_edge_set:
        bgnn_negative_edges.add(random_pair)
bgnn_negative_edges = list(bgnn_negative_edges)

# ── Split into training and validation sets (80/20) ──
bgnn_pos_train, bgnn_pos_val = train_test_split(bgnn_positive_edges, test_size=0.2, random_state=42)
bgnn_neg_train, bgnn_neg_val = train_test_split(bgnn_negative_edges, test_size=0.2, random_state=42)

# Convert to indexed pairs with labels: (source_idx, target_idx, label)
bgnn_train_indexed_pairs = (
    [(node_id_to_index[src], node_id_to_index[dst], 1) for (src, dst) in bgnn_pos_train] +
    [(node_id_to_index[src], node_id_to_index[dst], 0) for (src, dst) in bgnn_neg_train]
)
bgnn_val_indexed_pairs = (
    [(node_id_to_index[src], node_id_to_index[dst], 1) for (src, dst) in bgnn_pos_val] +
    [(node_id_to_index[src], node_id_to_index[dst], 0) for (src, dst) in bgnn_neg_val]
)

# ── Convert to tensors and move to the compute device ──
train_node_pairs_tensor = torch.tensor(
    [[src_idx, tgt_idx] for src_idx, tgt_idx, _ in bgnn_train_indexed_pairs], dtype=torch.long
).to(compute_device)
train_labels_tensor = torch.tensor(
    [label for _, _, label in bgnn_train_indexed_pairs], dtype=torch.long
).to(compute_device)

val_node_pairs_tensor = torch.tensor(
    [[src_idx, tgt_idx] for src_idx, tgt_idx, _ in bgnn_val_indexed_pairs], dtype=torch.long
).to(compute_device)
val_labels_tensor = torch.tensor(
    [label for _, _, label in bgnn_val_indexed_pairs], dtype=torch.long
).to(compute_device)

node_features_tensor = torch.tensor(node_feature_matrix, dtype=torch.float).to(compute_device)
graph_edge_index = graph_edge_index.to(compute_device)


# ============================================================
# Define the Bayesian GCN model for link prediction
# ============================================================
class BayesianGraphConvNetwork(torch.nn.Module):
    """
    A GCN with one graph convolution layer followed by a Bayesian
    linear classifier for edge-level binary classification.

    The forward pass:
      1. Apply GCNConv to produce node embeddings
      2. ReLU activation + dropout for regularization
      3. For each (source, target) pair, concatenate their embeddings
      4. Pass the concatenated edge features through a BayesLinear layer
    """

    def __init__(self, input_feature_dim, hidden_layer_dim, dropout_rate):
        super().__init__()
        # Graph convolution: transforms node features to hidden representations
        self.graph_conv_layer = pyg_nn.GCNConv(input_feature_dim, hidden_layer_dim)
        # Bayesian linear layer: models uncertainty in edge classification
        self.bayesian_classifier = bnn.BayesLinear(
            prior_mu=0, prior_sigma=0.1,
            in_features=hidden_layer_dim * 2,  # concatenation of two node embeddings
            out_features=2                      # binary classification: link / no-link
        )
        self.dropout_layer = torch.nn.Dropout(dropout_rate)

    def forward(self, node_features, edge_index, node_pair_indices):
        # Step 1: Message passing via GCN convolution
        node_embeddings = self.graph_conv_layer(node_features, edge_index)
        node_embeddings = torch.relu(node_embeddings)
        node_embeddings = self.dropout_layer(node_embeddings)

        # Step 2: Extract source and target embeddings for each pair
        source_embeddings = node_embeddings[node_pair_indices[:, 0]]
        target_embeddings = node_embeddings[node_pair_indices[:, 1]]

        # Step 3: Concatenate source and target features as edge representation
        concatenated_edge_features = torch.cat([source_embeddings, target_embeddings], dim=1)

        # Step 4: Classify through the Bayesian linear layer
        classification_logits = self.bayesian_classifier(concatenated_edge_features)
        return classification_logits


# ============================================================
# Optuna hyperparameter optimization
# ============================================================
def bgnn_optuna_objective(trial):
    """
    Objective function for Optuna: trains a BayesianGraphConvNetwork
    with trial-suggested hyperparameters and returns validation accuracy.
    """
    # Sample hyperparameters
    hidden_dim_candidate = trial.suggest_int('hidden_dim', 8, 64)
    dropout_candidate = trial.suggest_float('dropout', 0.1, 0.5)
    learning_rate_candidate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    # Initialize model, optimizer, and loss function
    trial_model = BayesianGraphConvNetwork(
        input_feature_dim=4,
        hidden_layer_dim=hidden_dim_candidate,
        dropout_rate=dropout_candidate
    ).to(compute_device)
    trial_optimizer = torch.optim.Adam(trial_model.parameters(), lr=learning_rate_candidate)
    cross_entropy_loss = torch.nn.CrossEntropyLoss()

    # Training loop (10 epochs for quick evaluation)
    trial_model.train()
    for epoch_idx in range(10):
        trial_optimizer.zero_grad()
        prediction_logits = trial_model(node_features_tensor, graph_edge_index, train_node_pairs_tensor)
        batch_loss = cross_entropy_loss(prediction_logits, train_labels_tensor)
        batch_loss.backward()
        trial_optimizer.step()

        # Log progress every 2 epochs
        if epoch_idx % 2 == 0:
            predicted_classes = prediction_logits.argmax(dim=1)
            train_accuracy = (predicted_classes == train_labels_tensor).float().mean().item()
            print(f'Trial {trial.number}, Epoch {epoch_idx}, '
                  f'Loss {batch_loss.item():.4f}, Train Acc {train_accuracy:.4f}')

    # Evaluate on validation set
    trial_model.eval()
    with torch.no_grad():
        val_logits = trial_model(node_features_tensor, graph_edge_index, val_node_pairs_tensor)
        val_predicted_classes = val_logits.argmax(dim=1)
        val_accuracy = (val_predicted_classes == val_labels_tensor).float().mean().item()

    return val_accuracy


# Run Optuna study to find the best hyperparameters
optuna_study = optuna.create_study(direction='maximize')
optuna_study.optimize(bgnn_optuna_objective, n_trials=3)  # fewer trials for speed
print('Best hyperparameters:', optuna_study.best_params)

# ============================================================
# Train the final BGNN model with the best hyperparameters
# ============================================================
best_hyperparams = optuna_study.best_params
final_bgnn_model = BayesianGraphConvNetwork(
    input_feature_dim=4,
    hidden_layer_dim=best_hyperparams['hidden_dim'],
    dropout_rate=best_hyperparams['dropout']
).to(compute_device)
final_optimizer = torch.optim.Adam(final_bgnn_model.parameters(), lr=best_hyperparams['lr'])
final_loss_function = torch.nn.CrossEntropyLoss()

for epoch_idx in range(10):
    final_optimizer.zero_grad()
    prediction_logits = final_bgnn_model(node_features_tensor, graph_edge_index, train_node_pairs_tensor)
    epoch_loss = final_loss_function(prediction_logits, train_labels_tensor)
    epoch_loss.backward()
    final_optimizer.step()

    if epoch_idx % 2 == 0:
        predicted_classes = prediction_logits.argmax(dim=1)
        train_accuracy = (predicted_classes == train_labels_tensor).float().mean().item()
        print(f'Final Model Epoch {epoch_idx}, Loss {epoch_loss.item():.4f}, '
              f'Train Acc {train_accuracy:.4f}')

# ── Evaluate final model on validation set ──
final_bgnn_model.eval()
with torch.no_grad():
    val_logits = final_bgnn_model(node_features_tensor, graph_edge_index, val_node_pairs_tensor)
    val_predicted_classes = val_logits.argmax(dim=1)
    final_val_accuracy = (val_predicted_classes == val_labels_tensor).float().mean().item()
print(f'Validation accuracy: {final_val_accuracy:.4f}')

# ============================================================
# Generate predictions on the test set and save submission
# ============================================================
bgnn_test_dataframe = pd.read_csv('test.csv').sort_values('Id')
test_node_pairs_tensor = torch.tensor(
    [[node_id_to_index[int(row.From)], node_id_to_index[int(row.To)]]
     for row in bgnn_test_dataframe.itertuples(index=False)],
    dtype=torch.long
).to(compute_device)

with torch.no_grad():
    test_logits = final_bgnn_model(node_features_tensor, graph_edge_index, test_node_pairs_tensor)
    # Convert logits to probabilities via softmax; take probability of class 1 (link exists)
    test_link_probabilities = torch.softmax(test_logits, dim=1)[:, 1].cpu().numpy()

bgnn_submission = pd.DataFrame({
    'Id': bgnn_test_dataframe['Id'].values,
    'Predictions': test_link_probabilities
})
bgnn_submission.to_csv('submission_bgnn.csv', index=False)
print('Saved: submission_bgnn.csv')
print(bgnn_submission.head())


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


...

Trial 0, Epoch 8, Loss 1064.1427, Train Acc 0.3735
Trial 1, Epoch 0, Loss 602.0149, Train Acc 0.4795
Trial 1, Epoch 2, Loss 439.4652, Train Acc 0.4938
Trial 1, Epoch 4, Loss 818.4416, Train Acc 0.3358
Trial 1, Epoch 6, Loss 1534.4448, Train Acc 0.3500
Trial 1, Epoch 8, Loss 1007.0773, Train Acc 0.3545


[I 2026-03-03 17:52:33,268] Trial 1 finished with value: 0.5 and parameters: {'hidden_dim': 13, 'dropout': 0.16949124775115038, 'lr': 0.00238661279715747}. Best is trial 0 with value: 0.5.


Trial 2, Epoch 0, Loss 309.3966, Train Acc 0.6298
Trial 2, Epoch 2, Loss 227.5949, Train Acc 0.5702
Trial 2, Epoch 4, Loss 417.4685, Train Acc 0.3806
Trial 2, Epoch 6, Loss 560.0106, Train Acc 0.5568


[I 2026-03-03 17:52:34,252] Trial 2 finished with value: 0.4065000116825104 and parameters: {'hidden_dim': 18, 'dropout': 0.29190261214087876, 'lr': 0.005358462327761068}. Best is trial 0 with value: 0.5.


Trial 2, Epoch 8, Loss 465.6781, Train Acc 0.5968
Best hyperparameters: {'hidden_dim': 16, 'dropout': 0.2821239500217849, 'lr': 0.0010413566045930896}
Final Model Epoch 0, Loss 1518.1907, Train Acc 0.4104
Final Model Epoch 2, Loss 1830.5684, Train Acc 0.4179
Final Model Epoch 4, Loss 718.9690, Train Acc 0.4663
Final Model Epoch 6, Loss 783.6557, Train Acc 0.4774
Final Model Epoch 8, Loss 803.9866, Train Acc 0.4601
Validation accuracy: 0.5000
Saved: submission_bgnn.csv
   Id  Predictions
0   1          1.0
1   2          1.0
2   3          1.0
3   4          1.0
4   5          1.0


## Model 3: Advanced Ensemble Pipeline

### Step 10: High-Performance Bagged Ensemble

**Summary:** This cell implements the full advanced link-prediction pipeline:

1. **PageRank computation** — Power-iteration PageRank (damping=0.85, 100 iterations) for global node importance scores.
2. **SVD embeddings** — 64-dimensional latent embeddings from the adjacency matrix via truncated SVD.
3. **77 hand-crafted features** — Combines structural features (degree, common neighbors, Jaccard, Adamic-Adar, preferential attachment), PageRank features, SVD dot-product/cosine/L2 embeddings, and cross-feature interactions.
4. **Bagged ensemble** — 7 rounds of bagging with fresh negative samples each round, training **LightGBM** (1500 trees, lr=0.02), **XGBoost** (700 trees, lr=0.03), and **Logistic Regression** independently.
5. **Grid-based weight optimization** — Searches over all weight combinations to find the optimal ensemble blend maximizing validation AUC.
6. **Submission generation** — Produces `submission_ensemble.csv` with the blended predictions.

In [ ]:
# =============================================================================
# PML Link Prediction — v6c: Restore exact v6 params + 150K training edges
#
# Reverted v6b changes that HURT score (0.9099 → 0.9060):
#   1. LightGBM: back to 1500 trees, lr=0.02 (was 2000/0.015 in v6b)
#   2. XGBoost: back to 1000 trees, early_stopping=30 (was 1200/50 in v6b)
#   3. Removed 3 redundant features (Katz, PR harmonic, CN median) → 77 feats
# One safe improvement: 150K training edges (was 100K) → more training data
# =============================================================================

import torch
import torch.nn as nn
import torchbnn as bnn
import pandas as pd
import numpy as np
import random
from math import log, sqrt
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

compute_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {compute_device} | XGBoost: {XGBOOST_AVAILABLE} | LightGBM: {LIGHTGBM_AVAILABLE}')


# =============================================================================
# STEP 1: Compute PageRank scores
# =============================================================================
print('\n══ Computing PageRank ══')

sorted_node_ids = sorted(list(all_graph_nodes))
node_to_matrix_index = {node_id: idx for idx, node_id in enumerate(sorted_node_ids)}
total_node_count = len(sorted_node_ids)

transition_row_indices = []
transition_col_indices = []
transition_weights = []

for source_node in outgoing_neighbors:
    destination_set = outgoing_neighbors[source_node]
    if not destination_set:
        continue
    edge_transition_weight = 1.0 / len(destination_set)
    for dest_node in destination_set:
        transition_row_indices.append(node_to_matrix_index[dest_node])
        transition_col_indices.append(node_to_matrix_index[source_node])
        transition_weights.append(edge_transition_weight)

transition_matrix = csr_matrix(
    (transition_weights, (transition_row_indices, transition_col_indices)),
    shape=(total_node_count, total_node_count)
)

damping_factor = 0.85
pagerank_vector = np.ones(total_node_count) / total_node_count

for iteration in range(30):
    updated_pagerank = (damping_factor * transition_matrix.dot(pagerank_vector)
                        + (1 - damping_factor) / total_node_count)
    updated_pagerank /= updated_pagerank.sum()
    if np.abs(updated_pagerank - pagerank_vector).sum() < 1e-8:
        print(f'  Converged at iteration {iteration + 1}')
        break
    pagerank_vector = updated_pagerank

pagerank_scores = {sorted_node_ids[idx]: pagerank_vector[idx] for idx in range(total_node_count)}
print(f'PageRank done. Max score = {pagerank_vector.max():.6f}')


# =============================================================================
# STEP 2: Compute SVD embeddings (64 dims)
# =============================================================================
print('\n══ Computing SVD embeddings (64 dims) ══')

adjacency_row_indices = []
adjacency_col_indices = []
for source_node in outgoing_neighbors:
    for dest_node in outgoing_neighbors[source_node]:
        adjacency_row_indices.append(node_to_matrix_index[source_node])
        adjacency_col_indices.append(node_to_matrix_index[dest_node])

adjacency_matrix = csr_matrix(
    (np.ones(len(adjacency_row_indices)), (adjacency_row_indices, adjacency_col_indices)),
    shape=(total_node_count, total_node_count)
)

SVD_EMBEDDING_DIM = 64

try:
    left_singular_vectors, singular_values, right_singular_vectors_transposed = svds(
        adjacency_matrix.astype(np.float32), k=SVD_EMBEDDING_DIM
    )
    sqrt_singular_values = np.sqrt(singular_values)
    svd_source_embeddings = left_singular_vectors * sqrt_singular_values[np.newaxis, :]
    svd_destination_embeddings = right_singular_vectors_transposed.T * sqrt_singular_values[np.newaxis, :]
    SVD_AVAILABLE = True
    print(f'SVD: {SVD_EMBEDDING_DIM} dims, top singular values: {singular_values[-5:][::-1]}')
except Exception as svd_error:
    SVD_AVAILABLE = False
    print(f'SVD failed: {svd_error}')


# =============================================================================
# STEP 3: Feature engineering — 77 features (exact v6 feature set)
# =============================================================================

def compute_advanced_link_features(source_node, target_node):
    """
    Compute 77 advanced structural, topological, and latent features.
    57 base + 5 SVD + 6 interaction + 3 SVD-hadamard + 1 three-hop + 2 CN-degree + 2 balance/directed.
    """
    source_undirected = undirected_neighbors.get(source_node, set())
    target_undirected = undirected_neighbors.get(target_node, set())
    source_outgoing = outgoing_neighbors.get(source_node, set())
    source_incoming = incoming_neighbors.get(source_node, set())
    target_outgoing = outgoing_neighbors.get(target_node, set())
    target_incoming = incoming_neighbors.get(target_node, set())

    source_out_deg = out_degree_map.get(source_node, 0)
    target_in_deg = in_degree_map.get(target_node, 0)
    source_in_deg = in_degree_map.get(source_node, 0)
    target_out_deg = out_degree_map.get(target_node, 0)
    source_undirected_deg = undirected_degree_map.get(source_node, 0)
    target_undirected_deg = undirected_degree_map.get(target_node, 0)

    # ── Common neighbor metrics (undirected) ──
    common_undirected_neighbors = source_undirected & target_undirected
    common_neighbor_count_und = len(common_undirected_neighbors)
    union_neighbor_count_und = len(source_undirected | target_undirected)

    jaccard_undirected = (common_neighbor_count_und / union_neighbor_count_und
                          if union_neighbor_count_und > 0 else 0.0)
    sorensen_dice_index = (2.0 * common_neighbor_count_und / (source_undirected_deg + target_undirected_deg)
                           if (source_undirected_deg + target_undirected_deg) > 0 else 0.0)
    hub_promoted_index = (common_neighbor_count_und / min(source_undirected_deg, target_undirected_deg)
                          if min(source_undirected_deg, target_undirected_deg) > 0 else 0.0)
    hub_depressed_index = (common_neighbor_count_und / max(source_undirected_deg, target_undirected_deg)
                           if max(source_undirected_deg, target_undirected_deg) > 0 else 0.0)
    cosine_similarity_und = (common_neighbor_count_und
                             / (sqrt(source_undirected_deg) * sqrt(target_undirected_deg))
                             if source_undirected_deg > 0 and target_undirected_deg > 0 else 0.0)

    # ── Directed common neighbor counts ──
    common_out_out = len(source_outgoing & target_outgoing)
    common_in_in = len(source_incoming & target_incoming)
    common_out_in = len(source_outgoing & target_incoming)
    common_in_out = len(source_incoming & target_outgoing)
    directed_two_path_count = common_out_in
    reverse_two_path_count = common_in_out

    # ── Adamic-Adar and Resource Allocation ──
    adamic_adar_undirected = sum(
        1.0 / log(undirected_degree_map.get(neighbor, 2))
        for neighbor in common_undirected_neighbors
        if undirected_degree_map.get(neighbor, 0) > 1
    )
    directed_common_neighbors = source_outgoing & target_incoming
    adamic_adar_directed = sum(
        1.0 / log(max(out_degree_map.get(neighbor, 0) + in_degree_map.get(neighbor, 0), 2))
        for neighbor in directed_common_neighbors
        if (out_degree_map.get(neighbor, 0) + in_degree_map.get(neighbor, 0)) > 1
    )
    resource_alloc_undirected = sum(
        1.0 / max(undirected_degree_map.get(neighbor, 1), 1)
        for neighbor in common_undirected_neighbors
    )
    resource_alloc_directed = sum(
        1.0 / max(out_degree_map.get(neighbor, 0) + in_degree_map.get(neighbor, 0), 1)
        for neighbor in directed_common_neighbors
    )

    # ── Preferential attachment ──
    pref_attach_directed = source_out_deg * target_in_deg
    pref_attach_undirected = source_undirected_deg * target_undirected_deg
    pref_attach_total = (source_out_deg + source_in_deg) * (target_out_deg + target_in_deg)

    reverse_edge_present = 1 if source_node in target_outgoing else 0

    # ── Directed Jaccard ──
    out_out_union = len(source_outgoing | target_outgoing)
    jaccard_out_out = common_out_out / out_out_union if out_out_union > 0 else 0.0
    in_in_union = len(source_incoming | target_incoming)
    jaccard_in_in = common_in_in / in_in_union if in_in_union > 0 else 0.0
    out_in_union = len(source_outgoing | target_incoming)
    jaccard_out_in = common_out_in / out_in_union if out_in_union > 0 else 0.0

    # ── Degree-derived ──
    source_total_degree = source_out_deg + source_in_deg
    target_total_degree = target_out_deg + target_in_deg
    source_out_in_ratio = source_out_deg / max(source_in_deg, 1)
    target_out_in_ratio = target_out_deg / max(target_in_deg, 1)
    source_common_fraction = common_neighbor_count_und / source_undirected_deg if source_undirected_deg > 0 else 0.0
    target_common_fraction = common_neighbor_count_und / target_undirected_deg if target_undirected_deg > 0 else 0.0

    # ── PageRank features ──
    source_pagerank = pagerank_scores.get(source_node, 1.0 / total_node_count)
    target_pagerank = pagerank_scores.get(target_node, 1.0 / total_node_count)
    pagerank_product = source_pagerank * target_pagerank

    transitivity_from_source = directed_two_path_count / source_out_deg if source_out_deg > 0 else 0.0
    transitivity_into_target = directed_two_path_count / target_in_deg if target_in_deg > 0 else 0.0

    source_out_overlap = common_out_out / source_out_deg if source_out_deg > 0 else 0.0
    target_out_overlap = common_out_out / target_out_deg if target_out_deg > 0 else 0.0
    source_in_overlap = common_in_in / source_in_deg if source_in_deg > 0 else 0.0
    target_in_overlap = common_in_in / target_in_deg if target_in_deg > 0 else 0.0

    # ── Base feature vector (57 features) ──
    feature_vector = [
        source_out_deg, target_in_deg, source_in_deg, target_out_deg,
        source_undirected_deg, target_undirected_deg,
        np.log1p(source_out_deg), np.log1p(target_in_deg),
        np.log1p(source_in_deg), np.log1p(target_out_deg),
        np.log1p(source_undirected_deg), np.log1p(target_undirected_deg),
        common_neighbor_count_und, np.log1p(common_neighbor_count_und),
        jaccard_undirected, sorensen_dice_index,
        hub_promoted_index, hub_depressed_index,
        cosine_similarity_und, adamic_adar_undirected,
        common_out_out, common_in_in, common_out_in, common_in_out,
        directed_two_path_count, reverse_two_path_count, np.log1p(directed_two_path_count),
        adamic_adar_directed, resource_alloc_undirected,
        resource_alloc_directed, jaccard_out_in,
        jaccard_out_out, jaccard_in_in,
        np.log1p(pref_attach_directed), np.log1p(pref_attach_undirected),
        np.log1p(pref_attach_total),
        reverse_edge_present,
        source_total_degree, target_total_degree,
        np.log1p(source_total_degree), np.log1p(target_total_degree),
        source_out_in_ratio, target_out_in_ratio,
        source_common_fraction, target_common_fraction,
        np.log1p(source_pagerank * 1e6), np.log1p(target_pagerank * 1e6),
        np.log1p(pagerank_product * 1e12),
        np.log1p((source_pagerank + target_pagerank) * 1e6),
        np.log(max(source_pagerank / max(target_pagerank, 1e-15), 1e-10)),
        max(source_pagerank, target_pagerank) * 1e6,
        min(source_pagerank, target_pagerank) * 1e6,
        transitivity_from_source, transitivity_into_target,
        source_out_overlap, target_out_overlap,
        source_in_overlap, target_in_overlap,
    ]

    # ── SVD embedding features (5 features) ──
    if SVD_AVAILABLE:
        source_matrix_idx = node_to_matrix_index.get(source_node, 0)
        target_matrix_idx = node_to_matrix_index.get(target_node, 0)
        source_svd_embedding = svd_source_embeddings[source_matrix_idx]
        target_svd_embedding = svd_destination_embeddings[target_matrix_idx]
        target_as_source_embedding = svd_source_embeddings[target_matrix_idx]
        source_as_dest_embedding = svd_destination_embeddings[source_matrix_idx]

        svd_forward_dot_product = np.dot(source_svd_embedding, target_svd_embedding)
        norm_source = np.linalg.norm(source_svd_embedding)
        norm_target = np.linalg.norm(target_svd_embedding)
        svd_forward_cosine = (svd_forward_dot_product / (norm_source * norm_target)
                              if norm_source > 0 and norm_target > 0 else 0.0)
        svd_forward_l2_distance = np.linalg.norm(source_svd_embedding - target_svd_embedding)
        svd_reverse_dot_product = np.dot(target_as_source_embedding, source_as_dest_embedding)

        feature_vector.extend([
            svd_forward_dot_product, svd_forward_cosine,
            svd_forward_l2_distance, svd_reverse_dot_product,
            svd_forward_dot_product - svd_reverse_dot_product,
        ])
    else:
        svd_forward_dot_product = 0.0
        svd_forward_cosine = 0.0
        source_svd_embedding = target_svd_embedding = np.zeros(SVD_EMBEDDING_DIM)
        feature_vector.extend([0.0] * 5)

    # ── Interaction features (6 features) ──
    feature_vector.extend([
        svd_forward_dot_product * np.log1p(common_neighbor_count_und) if SVD_AVAILABLE else 0.0,
        svd_forward_cosine * jaccard_undirected if SVD_AVAILABLE else 0.0,
        np.log1p(pagerank_product * 1e12) * jaccard_undirected,
        np.log1p(pagerank_product * 1e12) * np.log1p(common_neighbor_count_und),
        np.log1p(directed_two_path_count) * np.log1p(pref_attach_directed),
        adamic_adar_undirected * (1 + reverse_edge_present),
    ])

    # ── Advanced features (9 features) ──
    if SVD_AVAILABLE:
        svd_hadamard_product = source_svd_embedding * target_svd_embedding
        feature_vector.extend([
            np.mean(svd_hadamard_product),
            np.max(svd_hadamard_product),
            np.linalg.norm(source_svd_embedding - target_svd_embedding, ord=1),
        ])
    else:
        feature_vector.extend([0.0, 0.0, 0.0])

    three_hop_path_count = 0
    neighbor_cap = 50
    source_outgoing_capped = list(source_outgoing)[:neighbor_cap]
    for intermediate_node in source_outgoing_capped:
        intermediate_outgoing = outgoing_neighbors.get(intermediate_node, set())
        three_hop_path_count += len(intermediate_outgoing & target_incoming)
    feature_vector.append(np.log1p(three_hop_path_count))

    if common_undirected_neighbors:
        common_neighbor_degrees = [undirected_degree_map.get(neighbor, 0)
                                   for neighbor in common_undirected_neighbors]
        feature_vector.extend([
            np.mean(common_neighbor_degrees),
            np.max(common_neighbor_degrees),
        ])
    else:
        feature_vector.extend([0.0, 0.0])

    degree_balance_score = abs(source_out_deg - target_in_deg) / (source_out_deg + target_in_deg + 1)
    feature_vector.append(degree_balance_score)

    directed_cn_fraction = common_out_in / max(common_neighbor_count_und, 1)
    feature_vector.append(directed_cn_fraction)

    return feature_vector


# =============================================================================
# STEP 4: Generate PURE random negative samples
# =============================================================================
full_edge_set = set(directed_edge_list)
all_node_ids_list = list(all_graph_nodes)


def generate_random_negative_edges(num_negatives, random_seed=42):
    random_generator = random.Random(random_seed)
    negative_edge_set = set()
    while len(negative_edge_set) < num_negatives:
        random_source = random_generator.choice(all_node_ids_list)
        random_target = random_generator.choice(all_node_ids_list)
        if (random_source != random_target
                and (random_source, random_target) not in full_edge_set
                and (random_source, random_target) not in negative_edge_set):
            negative_edge_set.add((random_source, random_target))
    return list(negative_edge_set)


# =============================================================================
# STEP 5: Build training and validation datasets (150K edges — was 100K in v6)
# =============================================================================
print('\n══ Building training data (150K edges, PURE RANDOM negatives) ══')

TRAINING_EDGE_COUNT = 150000

sampled_positive_edges = directed_edge_list.copy()
random.shuffle(sampled_positive_edges)
sampled_positive_edges = sampled_positive_edges[:TRAINING_EDGE_COUNT]

sampled_negative_edges = generate_random_negative_edges(TRAINING_EDGE_COUNT, random_seed=42)
print(f'Positive edges: {len(sampled_positive_edges)}, Negative edges: {len(sampled_negative_edges)}')

positive_train_split, positive_val_split = train_test_split(
    sampled_positive_edges, test_size=0.15, random_state=42
)
negative_train_split, negative_val_split = train_test_split(
    sampled_negative_edges, test_size=0.15, random_state=42
)

ensemble_train_data = (
    [(src, tgt, 1) for src, tgt in positive_train_split] +
    [(src, tgt, 0) for src, tgt in negative_train_split]
)
ensemble_val_data = (
    [(src, tgt, 1) for src, tgt in positive_val_split] +
    [(src, tgt, 0) for src, tgt in negative_val_split]
)

print('Computing training features...')
ensemble_train_features = np.array(
    [compute_advanced_link_features(src, tgt) for src, tgt, _ in ensemble_train_data],
    dtype=np.float32
)
ensemble_train_labels = np.array(
    [label for _, _, label in ensemble_train_data], dtype=np.int64
)

ensemble_val_features = np.array(
    [compute_advanced_link_features(src, tgt) for src, tgt, _ in ensemble_val_data],
    dtype=np.float32
)
ensemble_val_labels = np.array(
    [label for _, _, label in ensemble_val_data], dtype=np.int64
)

feature_scaler = StandardScaler()
ensemble_train_features_scaled = feature_scaler.fit_transform(ensemble_train_features)
ensemble_val_features_scaled = feature_scaler.transform(ensemble_val_features)

print(f'Training samples: {len(ensemble_train_features)}, '
      f'Validation samples: {len(ensemble_val_features)}, '
      f'Feature count: {ensemble_train_features.shape[1]}')


# =============================================================================
# STEP 6: Train individual models (exact v6 hyperparameters)
# =============================================================================
print('\n══ Training models ══')

# ── Logistic Regression ──
best_logistic_auc = 0
best_logistic_C_value = 1
logistic_val_predictions = None

for regularization_C in [0.01, 0.1, 1, 10]:
    candidate_logistic_model = LogisticRegression(max_iter=3000, C=regularization_C, solver='lbfgs')
    candidate_logistic_model.fit(ensemble_train_features_scaled, ensemble_train_labels)
    candidate_val_probs = candidate_logistic_model.predict_proba(ensemble_val_features_scaled)[:, 1]
    candidate_auc = roc_auc_score(ensemble_val_labels, candidate_val_probs)
    if candidate_auc > best_logistic_auc:
        best_logistic_auc = candidate_auc
        best_logistic_C_value = regularization_C
        logistic_val_predictions = candidate_val_probs

print(f'Logistic Regression val AUC: {best_logistic_auc:.4f} (C={best_logistic_C_value})')

# ── LightGBM: 1500 trees, lr=0.02 (exact v6 params) ──
lightgbm_val_predictions = None
if LIGHTGBM_AVAILABLE:
    lightgbm_model = LGBMClassifier(
        n_estimators=1500, max_depth=8, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.7, min_child_samples=10,
        num_leaves=63, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbosity=-1, n_jobs=-1
    )
    lightgbm_model.fit(ensemble_train_features_scaled, ensemble_train_labels)
    lightgbm_val_predictions = lightgbm_model.predict_proba(ensemble_val_features_scaled)[:, 1]
    print(f'LightGBM val AUC: {roc_auc_score(ensemble_val_labels, lightgbm_val_predictions):.4f}')

# ── XGBoost: 1000 trees, lr=0.02, early_stopping=30 (exact v6 params) ──
xgboost_val_predictions = None
if XGBOOST_AVAILABLE:
    xgboost_model = XGBClassifier(
        n_estimators=1000, max_depth=7, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        gamma=0.1, eval_metric='auc', random_state=42,
        early_stopping_rounds=30, verbosity=0, n_jobs=-1
    )
    xgboost_model.fit(
        ensemble_train_features_scaled, ensemble_train_labels,
        eval_set=[(ensemble_val_features_scaled, ensemble_val_labels)],
        verbose=False
    )
    xgboost_val_predictions = xgboost_model.predict_proba(ensemble_val_features_scaled)[:, 1]
    print(f'XGBoost val AUC: {roc_auc_score(ensemble_val_labels, xgboost_val_predictions):.4f}')


# =============================================================================
# STEP 7: Bagged ensemble — 7 rounds
# =============================================================================
print('\n══ Bagged ensemble (7 rounds) ══')
NUM_BAGGING_ROUNDS = 7

ensemble_test_dataframe = pd.read_csv('test.csv').sort_values('Id')
print('Computing test features...')
test_feature_matrix_raw = np.array(
    [compute_advanced_link_features(int(row.From), int(row.To))
     for row in ensemble_test_dataframe.itertuples(index=False)],
    dtype=np.float32
)
print(f'Test feature matrix shape: {test_feature_matrix_raw.shape}')

bagged_lightgbm_predictions = []
bagged_xgboost_predictions = []
bagged_logistic_predictions = []

for bag_round in range(NUM_BAGGING_ROUNDS):
    bag_random_seed = 42 + bag_round * 17
    print(f'\n--- Bag {bag_round + 1}/{NUM_BAGGING_ROUNDS} (seed={bag_random_seed}) ---')

    bag_negative_edges = generate_random_negative_edges(TRAINING_EDGE_COUNT, random_seed=bag_random_seed)
    bag_labeled_pairs = (
        [(src, tgt, 1) for src, tgt in sampled_positive_edges] +
        [(src, tgt, 0) for src, tgt in bag_negative_edges]
    )
    random.shuffle(bag_labeled_pairs)

    bag_feature_matrix = np.array(
        [compute_advanced_link_features(src, tgt) for src, tgt, _ in bag_labeled_pairs],
        dtype=np.float32
    )
    bag_label_vector = np.array([label for _, _, label in bag_labeled_pairs], dtype=np.int64)

    bag_scaler = StandardScaler()
    bag_features_scaled = bag_scaler.fit_transform(bag_feature_matrix)
    bag_test_features_scaled = bag_scaler.transform(test_feature_matrix_raw)

    # ── LightGBM bag (1500 trees, lr=0.02) ──
    if LIGHTGBM_AVAILABLE:
        bag_lgbm_model = LGBMClassifier(
            n_estimators=1500, max_depth=8, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_samples=10,
            num_leaves=63, reg_alpha=0.1, reg_lambda=1.0,
            random_state=bag_random_seed, verbosity=-1, n_jobs=-1
        )
        bag_lgbm_model.fit(bag_features_scaled, bag_label_vector)
        bag_lgbm_probs = bag_lgbm_model.predict_proba(bag_test_features_scaled)[:, 1]
        bagged_lightgbm_predictions.append(bag_lgbm_probs)
        print(f'  LightGBM bag: mean={bag_lgbm_probs.mean():.4f}, std={bag_lgbm_probs.std():.4f}')

    # ── XGBoost bag (1000 trees) ──
    if XGBOOST_AVAILABLE:
        bag_xgb_model = XGBClassifier(
            n_estimators=1000, max_depth=7, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
            gamma=0.1, eval_metric='auc', random_state=bag_random_seed,
            verbosity=0, n_jobs=-1
        )
        bag_xgb_model.fit(bag_features_scaled, bag_label_vector)
        bag_xgb_probs = bag_xgb_model.predict_proba(bag_test_features_scaled)[:, 1]
        bagged_xgboost_predictions.append(bag_xgb_probs)
        print(f'  XGBoost bag:  mean={bag_xgb_probs.mean():.4f}, std={bag_xgb_probs.std():.4f}')

    # ── Logistic Regression bag ──
    bag_lr_model = LogisticRegression(max_iter=3000, C=best_logistic_C_value, solver='lbfgs')
    bag_lr_model.fit(bag_features_scaled, bag_label_vector)
    bag_lr_probs = bag_lr_model.predict_proba(bag_test_features_scaled)[:, 1]
    bagged_logistic_predictions.append(bag_lr_probs)
    print(f'  LogReg bag:   mean={bag_lr_probs.mean():.4f}, std={bag_lr_probs.std():.4f}')


# =============================================================================
# STEP 8: Ensemble weight optimization (step=0.025)
# =============================================================================
print('\n══ Final ensemble ══')

averaged_lightgbm_preds = (np.mean(bagged_lightgbm_predictions, axis=0)
                           if bagged_lightgbm_predictions else bagged_logistic_predictions[0])
averaged_xgboost_preds = (np.mean(bagged_xgboost_predictions, axis=0)
                          if bagged_xgboost_predictions else bagged_logistic_predictions[0])
averaged_logistic_preds = np.mean(bagged_logistic_predictions, axis=0)

print(f'LightGBM-bag avg: mean={averaged_lightgbm_preds.mean():.4f}')
print(f'XGBoost-bag avg:  mean={averaged_xgboost_preds.mean():.4f}')
print(f'LogReg-bag avg:   mean={averaged_logistic_preds.mean():.4f}')

print('\n── Weight optimization (3 models, step=0.025) ──')

model_val_predictions = {
    'LightGBM': lightgbm_val_predictions if LIGHTGBM_AVAILABLE else logistic_val_predictions,
    'XGBoost': xgboost_val_predictions if XGBOOST_AVAILABLE else logistic_val_predictions,
    'LogisticRegression': logistic_val_predictions
}
model_names_list = list(model_val_predictions.keys())
val_prediction_arrays = [model_val_predictions[name] for name in model_names_list]

optimal_val_auc = 0
optimal_ensemble_weights = None

for weight_lightgbm in np.arange(0.0, 1.001, 0.025):
    for weight_xgboost in np.arange(0.0, 1.001 - weight_lightgbm, 0.025):
        weight_logistic = 1.0 - weight_lightgbm - weight_xgboost
        if weight_logistic < -0.001 or weight_logistic > 1.001:
            continue
        weight_logistic = max(weight_logistic, 0.0)

        weighted_val_preds = (weight_lightgbm * val_prediction_arrays[0]
                              + weight_xgboost * val_prediction_arrays[1]
                              + weight_logistic * val_prediction_arrays[2])
        weighted_val_auc = roc_auc_score(ensemble_val_labels, weighted_val_preds)

        if weighted_val_auc > optimal_val_auc:
            optimal_val_auc = weighted_val_auc
            optimal_ensemble_weights = (weight_lightgbm, weight_xgboost, weight_logistic)

print('Best weights: ' + ', '.join(
    f'{name}={weight:.3f}'
    for name, weight in zip(model_names_list, optimal_ensemble_weights)
))
print(f'Best validation AUC: {optimal_val_auc:.4f}')

test_prediction_arrays = [averaged_lightgbm_preds, averaged_xgboost_preds, averaged_logistic_preds]
final_link_probabilities = sum(
    weight * predictions
    for weight, predictions in zip(optimal_ensemble_weights, test_prediction_arrays)
)
final_link_probabilities = np.clip(final_link_probabilities, 0, 1)

final_submission_dataframe = pd.DataFrame({
    'Id': ensemble_test_dataframe['Id'].values,
    'Predictions': final_link_probabilities
})
final_submission_dataframe.to_csv('submission_ensemble_v6c.csv', index=False)

print(f'\nSaved: submission_ensemble_v6c.csv')
print(f'Prediction stats: mean={final_link_probabilities.mean():.4f}, '
      f'std={final_link_probabilities.std():.4f}')
print(f'Range: [{final_link_probabilities.min():.4f}, {final_link_probabilities.max():.4f}]')

for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    count_above = (final_link_probabilities > threshold).sum()
    print(f'  P > {threshold}: {count_above} / {len(final_link_probabilities)}')

print(final_submission_dataframe.head(10))

Device: cuda | XGBoost: True | LightGBM: True

══ Computing PageRank ══
PageRank done. Max score = 0.405306

══ Computing SVD embeddings (64 dims) ══
SVD: 64 dims, top singular values: [1486.745    903.37305  758.77765  627.8559   608.10004]

══ Building training data (100K edges, PURE RANDOM negatives) ══
Positive edges: 100000, Negative edges: 100000
Computing training features...
Training samples: 170000, Validation samples: 30000, Feature count: 80

══ Training models ══
Logistic Regression val AUC: 1.0000 (C=1)


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM val AUC: 1.0000
XGBoost val AUC: 1.0000

══ Bagged ensemble (7 rounds) ══
Computing test features...
Test feature matrix shape: (2000, 80)

--- Bag 1/7 (seed=42) ---


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LightGBM bag: mean=0.9660, std=0.1501
  XGBoost bag:  mean=0.9533, std=0.1522
  LogReg bag:   mean=0.9886, std=0.0488

--- Bag 2/7 (seed=59) ---


...

  LightGBM bag: mean=0.9648, std=0.1400
  XGBoost bag:  mean=0.9417, std=0.1633
  LogReg bag:   mean=0.9894, std=0.0470

--- Bag 6/7 (seed=127) ---


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LightGBM bag: mean=0.9735, std=0.1163
  XGBoost bag:  mean=0.9543, std=0.1411
  LogReg bag:   mean=0.9872, std=0.0505

--- Bag 7/7 (seed=144) ---


c:\Users\10314026\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  LightGBM bag: mean=0.9809, std=0.0919
  XGBoost bag:  mean=0.9549, std=0.1378
  LogReg bag:   mean=0.9858, std=0.0615

══ Final ensemble ══
LightGBM-bag avg: mean=0.9694
XGBoost-bag avg:  mean=0.9493
LogReg-bag avg:   mean=0.9853

── Weight optimization (3 models, step=0.025) ──
Best weights: LightGBM=0.375, XGBoost=0.200, LogisticRegression=0.425
Best validation AUC: 1.0000

Saved: submission_ensemble_v6b.csv
Prediction stats: mean=0.9721, std=0.0853
Range: [0.2707, 1.0000]
  P > 0.1: 2000 / 2000
  P > 0.3: 1998 / 2000
  P > 0.5: 1983 / 2000
  P > 0.7: 1945 / 2000
  P > 0.9: 1837 / 2000
   Id  Predictions
0   1     0.999998
1   2     0.528941
2   3     0.997624
3   4     0.999894
4   5     0.999997
5   6     0.999493
6   7     0.999948
7   8     0.999938
8   9     0.985566
9  10     0.989831


---

## Notebook Summary — Link Prediction Pipeline

### Objective
Predict whether a directed edge (link) exists between two nodes in a large graph, framed as a **binary classification** problem.

---

### Pipeline Overview

| Step | Cell | Description |
|------|------|-------------|
| 1 | Imports | Load pandas, numpy, sklearn, torch, torch_geometric, torchbnn, optuna, LightGBM, XGBoost |
| 2 | Config | Set random seeds (42) and define `POSITIVE_EDGE_SAMPLE_LIMIT` |
| 3 | Graph Parsing | Read `train.csv`, build directed adjacency lists, compute degree dictionaries |
| 4 | Feature Function | Define `compute_basic_link_features()` — 13 structural features per node pair |
| 5 | Negative Sampling | Generate balanced positive/negative samples, split 80/20 train/val |
| 6 | Logistic Regression | Train StandardScaler + LogisticRegression pipeline, evaluate AUC-ROC |
| 7 | GridSearchCV | Tune LR hyperparameters (C, solver, max_iter) via 5-fold CV |
| 8 | Test Submission | Score test edges and save `submission.csv` |
| 9 | BGNN | Bayesian GCN with Optuna tuning for uncertainty-aware link prediction |
| 10 | Ensemble | PageRank + SVD + 77 features + bagged LightGBM/XGBoost/LR ensemble |

---

### Models Implemented

1. **Logistic Regression** — Simple linear baseline with StandardScaler and balanced class weights
2. **Bayesian Graph Neural Network (BGNN)** — GCNConv + BayesLinear layers, optimized via Optuna, provides uncertainty estimates
3. **Advanced Ensemble** — Bagged combination of LightGBM (1500 trees), XGBoost (700 trees), and Logistic Regression with grid-optimized blending weights

---

### Key Features Engineered

- **Structural:** Degree (in/out/undirected), common neighbors, Jaccard similarity, Adamic-Adar index, preferential attachment, reverse edge flag
- **Global:** PageRank scores (power iteration, damping=0.85)
- **Latent:** SVD-64 embeddings (dot product, cosine similarity, L2 distance)
- **Interaction:** Cross-feature products for richer signal capture
- **Total:** 13 basic features (Models 1–2), 77 advanced features (Model 3)

---

### Evaluation
- **Metric:** AUC-ROC (Area Under Receiver Operating Characteristic Curve)
- **Validation:** 80/20 train/val split with `random_state=42`
- **Cross-validation:** 5-fold CV for hyperparameter tuning
- **Bagging:** 7 rounds with re-sampled negatives for prediction stability

---

### Output Files
- `submission.csv` — Logistic Regression predictions
- `submission_bgnn.csv` — BGNN predictions
- `submission_ensemble.csv` — Advanced ensemble predictions